<center><h1>Wang_Selena_HW3</h1></center>
<br>
<br>

Name: Selena Wang
<br>
Github Username: slnwang
<br>
USC ID: 6791950498

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import bootstrap


Get the AReM Data Set

In [14]:
data_dir = Path("../data")

activities = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
columns = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

instances = {}

for activity in activities:
    folder = data_dir / activity

    for file in folder.glob("dataset*.csv"):
        num = int(file.stem.replace("dataset", ""))

        rows = []

        with open(file) as f:
            for line in f:
                line = line.strip()

                if line.startswith("#") or not line:
                    continue

                values = line.replace(",", " ").split()

                if len(values) == 7:
                    rows.append(values)

        df = pd.DataFrame(rows, columns=columns)
        df = df.apply(pd.to_numeric)

        instances[(activity, num)] = df

print("Total instances:", len(instances))
print(instances[("bending1", 1)].shape)
instances[("bending1", 1)].head()

Total instances: 88
(480, 7)


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0,39.25,0.43,22.75,0.43,33.75,1.3
1,250,39.25,0.43,23.00,0.00,33.00,0.0
2,500,39.25,0.43,23.25,0.43,33.00,0.0
3,750,39.50,0.50,23.00,0.71,33.00,0.0
4,1000,39.50,0.50,24.00,0.00,33.00,0.0


### (b) Test and Train Data

In [15]:
train_data = {}
test_data = {}

for (activity, num), df in instances.items():
    if activity in ["bending1", "bending2"]:
        if num <= 2:
            test_data[(activity, num)] = df
        else:
            train_data[(activity, num)] = df
    else:
        if num <= 3:
            test_data[(activity, num)] = df
        else:
            train_data[(activity, num)] = df

print("Training instances:", len(train_data))
print("Testing instances:", len(test_data))


Training instances: 69
Testing instances: 19


### (c) Feature Extraction

#### i. Research

* Mean: average of all values, captures overall signal level
* Median: middle value, captures central tendency (robust to outliers)
* Standard deviation: spread of values around the mean, captures variability
* Variance: average squared deviation from the mean, captures degree of fluctuation
* Minimum: smallest value, captures lower extreme
* Maximum: largest value, captures upper extreme
* Range: Max - Min, captures spread
* Q1/25th Percentile: 25% of observations fall below, captures lower distribution
* Q3/75th Percentile: 75% of observations fall below, captures upper distribution
* IQR/Interquartile range: Q3 - Q1, captures spread of middle 50% of observations
* Skewness: asymmetry of the distribution, captures direction and degree of asymmetry
* Kurtosis: tail heaviness of the distribution, captures tendency for extreme values or outliers
* Autocorrelation: relationship between observations at different time lags
* Mean absolute change: average absolute difference between consecutive observations
* Root Mean Square (RMS): measures overall magnitude of signal

Sources:

Christ, M., Braun, N., Neuffer, J., & Kempa-Liehr, A. W. (2018). Time series feature extraction on basis of scalable hypothesis tests (tsfresh – A Python package). *Neurocomputing, 307*, 72–77. https://doi.org/10.1016/j.neucom.2018.03.067

https://tsfresh.readthedocs.io/en/latest/text/list_of_features.html 

#### ii. Extraction

In [16]:
features = []

for (activity, num), df in instances.items():
    row = {"activity": activity, "dataset": num}

    for i, col in enumerate(columns[1:], 1):
        row[f"min{i}"] = df[col].min()
        row[f"max{i}"] = df[col].max()
        row[f"mean{i}"] = df[col].mean()
        row[f"median{i}"] = df[col].median()
        row[f"std{i}"] = df[col].std()
        row[f"q1_{i}"] = df[col].quantile(0.25)
        row[f"q3_{i}"] = df[col].quantile(0.75)

    features.append(row)

features_df = pd.DataFrame(features)

print(features_df.shape)
features_df.head()


(88, 44)


,activity,dataset,min1,max1,mean1,median1,std1,q1_1,q3_1,min2,...,std5,q1_5,q3_5,min6,max6,mean6,median6,std6,q1_6,q3_6
0,bending1,7,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,0.0,...,3.318301,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83
1,bending1,6,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,0.0,...,2.488862,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87
2,bending1,4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00
3,bending1,5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50
4,bending1,1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.00,1.30


#### iii. Standard Deviation

In [18]:
results = []

for col in features_df.columns[2:]:
    values = features_df[col].to_numpy()

    std = np.std(values, ddof=1)

    ci = bootstrap(
        (values,),
        lambda x: np.std(x, ddof=1),
        confidence_level=0.90,
        n_resamples=10000,
        method="percentile",
        random_state=42
    ).confidence_interval

    results.append({
        "Feature": col,
        "Std": std,
        "Lower CI": ci.low,
        "Upper CI": ci.high
    })

std_df = pd.DataFrame(results)
std_df


,Feature,Std,Lower CI,Upper CI
0,min1,9.569975,8.254117,10.776605
1,max1,4.394362,3.329673,5.307036
2,mean1,5.335718,4.703903,5.884731
3,median1,5.440054,4.799102,6.000340
4,std1,1.772153,1.571622,1.950517
5,q1_1,6.153590,5.575792,6.646527
6,q3_1,5.138925,4.312816,5.845452
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.631465,5.411143
9,mean2,1.574164,1.396109,1.706238


#### iv. Select Features

For the 3 important time-domain features, I would choose mean, standard deviation, and maximum:
* Mean: captures the average sensor reading and overall signal level, helping distinguish between the different activities
* Standard deviation: captures the variability of the sensor readings, helping differentiate activites with more movement from ones that may be more stationary
* Maximum: captures the highest sensor reading, helping identify activities with more extreme movements

Using these 3 together can help capture the central tendency, variability and extreme values, providing different characteristics of the time series that could be useful for classifying activities. 

## 2. ISLR 3.7.4

### (a) Linear Train

If the true relationship between X and Y is linear, we would expect for the training cubic regression RSS to be less than or equal to the training linear regression RSS. That is: $RSS_{Cubic} \leq RSS_{Linear}$

This is because the cubic regression model is more flexible than the linear and can fit the training data at least as well as  the linear regression model. In addition, the cubic model includes the linear regression model (when two of the betas = 0) so its training RSS can't be higher. 

### (b) Linear Test

If the true relationship between X and Y is linear, we would expect for the test linear regression RSS to be lower than the test cubic regression RSS.

Since the true relationship is linear, the cubic regression model could overfit by introducing unnecessary complexity while the linear model is correctly specified. The linear model would be expected to generalize better to new data and thus have a lower test RSS. 

### (c) Not Linear Train

If the true relationship between X and Y is not linear but we don't know how far from linear, we would still expect training cubic regression RSS to be less than or equal to the training linear regression RSS. That is: $RSS_{Cubic} \leq RSS_{Linear}$

This is because regardless of true relationship, the more flexible cubic regression allows us to fit more of the training data at least as well as the linear regression model. The polynomial terms allow the model to capture more patterns than the linear so we can expect training RSS for the cubic model to be lower or equal to the linear training RSS.

### (d) Not Linear Testing

If the true relationship between X and Y is not linear, but we don't know how far it is from linear, we do not have enough information to tell which model will have a lower test RSS.

If the true relationship is strongly nonlinear and can be approximated well by a cubic function, the cubic model may have a lower test RSS. However, if the relationship is only slightly nonlinear, the linear model may perform better due to its lower variance. Since we don't know how far the true relationship is from linear, we cannot determine which model will have the lower test RSS.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 